# 01 — Data Inventory & Provenance

**Objective:** Establish a traceable data asset map for the Belgium AED Optimization project.

This notebook documents:
1. All raw data files, their schemas, row counts, and temporal coverage
2. Structural inconsistencies across intervention files (column naming, date formats)
3. Rationale for the downstream cleaning strategy

In [3]:
from pathlib import Path
import pandas as pd
import re

# All paths are relative to the repository root
PROJECT_ROOT = Path('.')
RAW_DIR = PROJECT_ROOT / 'mda_project' / 'data' / 'raw'

print('Raw data directory:', RAW_DIR.resolve())
print('\nFiles available:')
for p in sorted(RAW_DIR.glob('*')):
    if p.is_file():
        size_mb = p.stat().st_size / (1024 * 1024)
        print(f'  {p.name:45s} {size_mb:>7.1f} MB')

Raw data directory: /Users/Zhuanz/S/已完成任务/mda 项目/mda_project/data/raw

Files available:
  BELGIUM_-_Provinces.geojson                       9.9 MB
  België.json                                      0.0 MB
  aed_locations.parquet.gzip                        0.3 MB
  ambulance_locations.parquet.gzip                  0.0 MB
  cad9.parquet.gzip                                17.3 MB
  interventions1.parquet.gzip                      27.8 MB
  interventions2.parquet.gzip                      28.7 MB
  interventions3.parquet.gzip                      27.9 MB
  interventions_bxl2.parquet.gzip                   1.7 MB
  mug_locations.parquet.gzip                        0.0 MB
  pit_locations.parquet.gzip                        0.0 MB


## 1.1 Intervention Files — Schema Comparison

The intervention data is split across 4 parquet files with potentially inconsistent column names.
We normalize column names and build a comparative inventory table.

In [4]:
def normalize_column(c: str) -> str:
    """Normalize column names to lowercase snake_case."""
    c = c.strip().lower()
    c = re.sub(r'[^a-z0-9]+', '_', c)
    return re.sub(r'_+', '_', c).strip('_')

inter_files = [
    'interventions1.parquet.gzip',
    'interventions2.parquet.gzip',
    'interventions3.parquet.gzip',
    'interventions_bxl2.parquet.gzip',
]

inventory = []
all_columns = set()

for fn in inter_files:
    fp = RAW_DIR / fn
    if not fp.exists():
        continue
    df = pd.read_parquet(fp)
    norm_cols = [normalize_column(c) for c in df.columns]
    all_columns.update(norm_cols)
    inventory.append({
        'File': fn,
        'Rows': f'{len(df):,}',
        'Columns': len(df.columns),
        'Sample Columns': ', '.join(norm_cols[:8]) + '...'
    })

inv_df = pd.DataFrame(inventory)
print(inv_df)
print(f'\nTotal unique normalized columns across all files: {len(all_columns)}')

,File,Rows,Columns,Sample Columns
0,interventions1.parquet.gzip,"200,627",46,"mission_id, service_name, postalcode_permanenc..."
1,interventions2.parquet.gzip,"200,627",46,"mission_id, service_name, postalcode_permanenc..."
2,interventions3.parquet.gzip,"200,627",46,"mission_id, service_name, postalcode_permanenc..."
3,interventions_bxl2.parquet.gzip,"38,620",36,"mission_id, t0, cityname_intervention, longitu..."



Total unique normalized columns across all files: 60


## 1.2 Temporal Format Diagnostic

A critical finding: the `T0` and `T3` timestamp columns use **different formats** across files,
which caused parsing failures in earlier pipeline versions.

In [5]:
for fn in inter_files:
    fp = RAW_DIR / fn
    if not fp.exists():
        continue
    df = pd.read_parquet(fp)
    df.columns = [normalize_column(c) for c in df.columns]
    t0_samples = df['t0'].dropna().astype(str).head(3).tolist() if 't0' in df.columns else ['N/A']
    t3_samples = df['t3'].dropna().astype(str).head(3).tolist() if 't3' in df.columns else ['N/A']
    print(f'\n{fn}')
    print(f'  T0 format samples: {t0_samples}')
    print(f'  T3 format samples: {t3_samples}')


interventions1.parquet.gzip
  T0 format samples: ['01JUN22:00:01:34', '01JUN22:00:03:51', '01JUN22:00:03:51']
  T3 format samples: ['2022-06-01 00:17:53.888', '2022-06-01 00:16:25.356', '2022-06-01 00:35:58.397']

interventions2.parquet.gzip
  T0 format samples: ['11JUL22:07:13:11', '11JUL22:07:14:34', '11JUL22:07:25:15']
  T3 format samples: ['2022-07-11 07:28:57.144', '2022-07-11 07:29:44.622', '2022-07-11 07:48:21.816']

interventions3.parquet.gzip
  T0 format samples: ['05JAN23:01:18:27', '05JAN23:01:20:59', '05JAN23:01:34:07']
  T3 format samples: ['2023-01-05 01:26:06.798', '2023-01-05 01:28:53.947', '2023-01-05 01:44:38.018']

interventions_bxl2.parquet.gzip
  T0 format samples: ['01JUN22:00:02:45', '01JUN22:00:04:58', '01JUN22:00:07:43']
  T3 format samples: ['01JUN22:00:15:59', '01JUN22:00:19:43', '01JUN22:00:20:29']


## 1.3 Supplementary Datasets

Beyond intervention records, the project uses several auxiliary spatial datasets.

In [6]:
aux_files = ['aed_locations.parquet.gzip', 'ambulance_locations.parquet.gzip',
             'mug_locations.parquet.gzip', 'cad9.parquet.gzip',
             'BELGIUM_-_Provinces.geojson']

aux_info = []
for fn in aux_files:
    fp = RAW_DIR / fn
    if not fp.exists():
        continue
    if fn.endswith('.geojson'):
        import geopandas as gpd
        gdf = gpd.read_file(fp)
        aux_info.append({'File': fn, 'Rows': len(gdf), 'Columns': len(gdf.columns), 'Type': 'GeoJSON'})
    else:
        df = pd.read_parquet(fp)
        aux_info.append({'File': fn, 'Rows': len(df), 'Columns': len(df.columns), 'Type': 'Parquet'})

print(pd.DataFrame(aux_info))

,File,Rows,Columns,Type
0,aed_locations.parquet.gzip,15227,11,Parquet
1,ambulance_locations.parquet.gzip,279,9,Parquet
2,mug_locations.parquet.gzip,94,10,Parquet
3,cad9.parquet.gzip,289401,35,Parquet
4,BELGIUM_-_Provinces.geojson,11,9,GeoJSON


## Summary

**Key findings from the data inventory:**

| Finding | Detail |
|---------|--------|
| Timestamp formats | T0 uses `DDMMMYY:HH:MM:SS` (e.g., `01JUN22:00:01:34`); T3 uses ISO format (e.g., `2022-06-01 00:17:53.888`) |
| Schema variation | `interventions_bxl2` has a different column schema than the other 3 files |
| Spatial datasets | AED locations, ambulance bases, MUG dispatch centers, and province boundaries are available |

These findings inform the cleaning strategy implemented in **Notebook 02**.